# Notebook 01 — Persistent Homology Analysis

**Goal:** Compute and compare persistence diagrams between 'accelerated' and 'resilient' groups for each omics layer.

We use synthetic data with Tian-assigned labels as a proof of concept, then compute:
1. Persistence diagrams per group per layer
2. Wasserstein/Bottleneck distances between groups
3. Statistical significance via permutation testing

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from data_utils import generate_synthetic_multimics, preprocess_omics, assign_groups_from_tian_score
from tda_utils import (
    compute_persistence_diagrams,
    compute_all_layers_dgms,
    wasserstein_distance,
    bottleneck_distance,
    permutation_test,
    diagnose_persistence,
)
from visualization import plot_barcode, plot_persistence_diagram, plot_barcode_comparison
from config import RANDOM_SEED

sns.set_style('whitegrid')
%matplotlib inline
print('✅ Imports OK')

## 1. Load & Prepare Data

In [ ]:
# Generate synthetic multi-omics with known topology
ds = generate_synthetic_multimics(n_samples=300, topology_type='circle', noise=0.08, n_features=50)

layers = {
    'transcriptomics': preprocess_omics(pd.DataFrame(ds['transcriptomics']), method='standard'),
    'metabolomics': preprocess_omics(pd.DataFrame(ds['metabolomics']), method='standard'),
    'epigenomics': preprocess_omics(pd.DataFrame(ds['epigenomics']), method='standard'),
}

metadata = ds['metadata']
labels = ds['labels']

print(f"Groups: {labels.value_counts().to_dict()}")
print(f"Layer shapes: {[(k, v.shape) for k, v in layers.items()]}")

## 2. Compute Persistence Diagrams per Group per Layer

In [ ]:
# Split data by group
accel_idx = labels == 'accelerated'
resil_idx = labels == 'resilient'

accel_layers = {k: v[accel_idx] for k, v in layers.items()}
resil_layers = {k: v[resil_idx] for k, v in layers.items()}

print('Computing persistence for ACCELERATED group...')
dgms_accel = compute_all_layers_dgms(accel_layers, max_dim=2)

print('\nComputing persistence for RESILIENT group...')
dgms_resil = compute_all_layers_dgms(resil_layers, max_dim=2)

print('\n✅ All diagrams computed')

## 3. Visual Comparison — H1 Barcodes

In [ ]:
for layer_name in layers.keys():
    fig = plot_barcode_comparison(
        dgms_accel[layer_name],
        dgms_resil[layer_name],
        labels=('Accelerated', 'Resilient'),
        dim=1,
    )
    fig.suptitle(f'H1 Barcode Comparison — {layer_name}', fontsize=14, fontweight='bold', y=1.02)
    plt.savefig(f'../results/figures/barcode_comparison_{layer_name}.png', dpi=150, bbox_inches='tight')
    plt.show()

## 4. Topological Distances Between Groups

In [ ]:
print(f"{'Layer':<25} {'Wasserstein H1':>15} {'Bottleneck H1':>15}")
print('-' * 55)

distance_table = []
for layer_name in layers.keys():
    w_dist = wasserstein_distance(dgms_accel[layer_name], dgms_resil[layer_name], dim=1)
    b_dist = bottleneck_distance(dgms_accel[layer_name], dgms_resil[layer_name], dim=1)
    distance_table.append({'layer': layer_name, 'wasserstein_H1': w_dist, 'bottleneck_H1': b_dist})
    print(f"{layer_name:<25} {w_dist:>15.4f} {b_dist:>15.4f}")

df_distances = pd.DataFrame(distance_table)
print(f"\n✅ Mean Wasserstein: {df_distances['wasserstein_H1'].mean():.4f}")

## 5. Permutation Test for Significance

In [ ]:
result = permutation_test(dgms_accel, dgms_resil, n_perm=500, dim=1)

print(f"Observed multi-view topological distance: {result['observed_distance']:.4f}")
print(f"p-value: {result['p_value']:.4f}")
print(f"Significant at α=0.05: {'✅ YES' if result['p_value'] < 0.05 else '❌ NO'}")

## 6. Diagnostic Summary

In [ ]:
for layer_name in layers.keys():
    print(f"\n{'='*60}")
    print(f"  {layer_name.upper()}")
    print(f"{'='*60}")
    
    for group_name, dgms in [('Accelerated', dgms_accel), ('Resilient', dgms_resil)]:
        diag = diagnose_persistence(dgms[layer_name])
        print(f"  {group_name}:")
        for dim, info in diag.items():
            print(f"    {dim}: n={info['n_features']}, "
                  f"mean_lifetime={info['mean_lifetime']:.4f}")

## 7. Key Findings

- The resilient group shows [more/fewer] persistent H1 features
- Wasserstein distance between groups: [value]
- Permutation test: [significant / not significant]
- **Biological interpretation:** H1 features may represent cyclical metabolic pathways that are preserved in resilient individuals